# Dense YOLO feature extractor

## Purpose
Dense YOLO feature extractor within the MLC Robotics language-conditioned YOLO policy programme.

## Environment and inputs
No attached checkpoint. GPU and Internet required.

## Execution
The code cells run sequentially using Run All. Embedded installation code provisions the required project modules.

## Outputs
Extractor graph, routing, feature and gradient checks.

## Evidence and interpretation
Component validation only; detector accuracy and task success are separate.

See docs/notebook_guides/validate_extractor_kaggle_standalone.md for the experiment record.


In [ ]:
import sys, platform, subprocess, json
from pathlib import Path
import torch, torchvision, numpy
print(platform.python_version())
print({"torch": torch.__version__, "torchvision": torchvision.__version__, "numpy": numpy.__version__, "cuda": torch.cuda.is_available()})
assert torch.cuda.is_available(), "Enable GPU before running the GPU validation."
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i))


In [ ]:
import base64, hashlib, io, tarfile
from pathlib import Path
# Embedded reviewed project; no separate Kaggle input is required.
PROJECT_BASE64 = '...'
project_bytes = base64.b64decode(PROJECT_BASE64)
assert hashlib.sha256(project_bytes).hexdigest() == '44cfd9ed6c8d05d29d00ae3477d37e32980aa5466b8307619353bfa98a7b8caf'
ROOT = Path('/kaggle/working/smol-vla-review')
ROOT.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(project_bytes), mode='r:gz') as archive:
    archive.extractall(ROOT, filter='data')
assert (ROOT / 'pyproject.toml').is_file()
assert (ROOT / 'tests/test_yolo_world_extractor.py').is_file()
print('Reviewed project extracted:', ROOT)


In [ ]:
SHA = 'aa1f778a75a610f7dd2aa4762df1c25f6a3063d7'
UPSTREAM = Path('/kaggle/working/ultralytics-source')
def run(args, **kwargs):
    return subprocess.run(args, check=True, **kwargs)
if not UPSTREAM.exists():
    run(['git', 'clone', '--no-checkout', 'https://github.com/ultralytics/ultralytics.git', str(UPSTREAM)])
run(['git', '-C', str(UPSTREAM), 'checkout', '--detach', SHA])
actual = subprocess.check_output(['git', '-C', str(UPSTREAM), 'rev-parse', 'HEAD'], text=True).strip()
assert actual == SHA
# Do not let pip replace the working PyTorch/CUDA stack.
run([sys.executable, '-m', 'pip', 'install', '--no-deps', '--no-build-isolation',
     str(UPSTREAM), 'pytest', 'ultralytics-thop'])
# If this reports a missing dependency, stop and share the error. Do not upgrade torch.
run([sys.executable, '-c',
     'import torch, torchvision, ultralytics; from ultralytics.nn.tasks import WorldModel; print(ultralytics.__version__); assert ultralytics.__version__ == "8.4.189"'])


In [ ]:
import os
LOG = ROOT / 'reports' / 'kaggle_extractor_validation.log'
XML = ROOT / 'reports' / 'kaggle_extractor_validation.xml'
env = dict(os.environ)
env['PYTHONPATH'] = str(ROOT / 'src')
env['OMP_NUM_THREADS'] = '2'
env['MKL_NUM_THREADS'] = '2'
# Most graph tests intentionally run on CPU; the marked smoke test runs on GPU.
cmd = [sys.executable, '-m', 'pytest', '-v', '-ra',
       'tests/test_yolo_world_extractor.py', f'--junitxml={XML}']
with LOG.open('w') as log:
    proc = subprocess.Popen(cmd, cwd=ROOT, env=env, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end='')
        log.write(line)
    result = proc.wait()
assert result == 0, f'Tests failed (exit {result}); share {LOG.name}.'
import xml.etree.ElementTree as ET
suites = ET.parse(XML).getroot()
assert sum(int(s.get('tests', 0)) for s in suites.iter('testsuite')) > 0
assert sum(int(s.get('skipped', 0)) for s in suites.iter('testsuite')) == 0, 'GPU validation must not skip tests.'
print('Extractor validation passed. Save the log and XML with this source version.')
